# LaGoVAD-KIM — arms **K0 / K0k9**, pilot KP1–KP4 on the decided control, then **K5** (§6.1), seed 2024

Rule (written before any number): `core/docs/final/KIM_RUN.md` (§3 arms, §3.1 K0k9, §5–§5.2 pilot, §6 + §6.1 read-out).
Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad`. **Nothing is extracted**: every input is a cache the
motion-stream host already read (A2 bakes with Nexar statistics, `DoTA_s1_ncc`). **W0's checkpoint no longer exists**:
W0 is only a printed row, from its saved reads and config (§4 dated note); the control is K0.

| step | what | GPU | estimate |
|---|---|---|---|
| 1–3 | setup, preflight | — | 5–10 min |
| 4 | pytest on the KIM code — **raises if red** | no | 2–5 min |
| 4b | W0's saved reads are on Drive (else upload them) | no | seconds |
| 5 | train **K0** (W0 recipe + DVS off) and **K0k9** (+ kernel 9) | **yes** | 2 × 15–45 min |
| 6 | K0 / K0k9: DoTA-CAP-dev (protocol B), DADA (whole), TAD | **yes** | 2 × 10–20 min |
| 7 | read-out 1 → **kernel decision (§3.1)**; paste `REPORTS/kim_readout_host.md` | no | 1 min |
| 8 | **pilot KP1–KP4** on the decided control → paste `REPORTS/pilot/kim_pilot.md` | yes | 10–20 min |
| 8b | **pilot again with the KP3 split** (first vs last pair) → paste `REPORTS/pilot_edges/kim_pilot.md` | yes | 10–20 min |
| 9 | train **K5** (`kim.edge_mask` if KP3 failed) on the decided host, read it | **yes** | 25–60 min |
| 10 | read-out 2 → **stop rule (§6)**; paste `REPORTS/kim_readout.md` | no | 1 min |

Every step skips what is already on Drive. One seed ⇒ **PRELIMINARY (n = 1)**. DVS is off in every K row
(`dvs.theta = dvs.theta_ego = 1`; `dvs_sup` still runs on normal bags, KIM_RUN.md D2).

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data, cache/clip
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: Nexar_window, A2 bakes, KNN, v2 A2 runs
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code + every output this notebook writes
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert 'decided control' in getattr(constants, 'KIM_PILOT_DISCLOSURE', ''), 'this upload predates the pilot move -- re-upload kat-vad/'
assert '### 5.3 The mask moves' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §5.3 in this upload -- re-upload kat-vad/'

SEED     = constants.KIM_SEED                      # 2024
ENC      = constants.V5_ENCODER                    # V2-S
C        = constants.NEXAR_CONSTRUCTION_WINDOW
DATASET  = constants.NEXAR_DATASET
T2NAME   = constants.DADA_ORIGIN_DATASET
CAP_DEV  = constants.V2_SPLIT_DOTA_CAP_DEV
W0       = constants.ZS_CONTROL_ARM                # 'W0'
K0, K0K9, K2, K5 = constants.KIM_ARMS              # ('K0', 'K0k9', 'K2', 'K5')

V2_CACHE, V2_OUT = Path(DRIVE_V2) / 'cache', Path(DRIVE_V2) / 'outputs'
BUILD    = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n4' / 'build_report.json'
DATA     = Path(DRIVE_V2) / 'data' / f'{DATASET}_{C}'
A2_NEXAR = V2_CACHE / 'v2' / 'A2_S' / 'Nexar_s8'
A2_DADA  = V2_CACHE / 'v2' / 'A2_S' / f'{T2NAME}_test_sources_nexar'
A2_DOTA  = V2_CACHE / 'v2' / 'A2_S' / f'{CAP_DEV}_s3_from_s1_nexar'
A2_TAD   = V2_CACHE / 'v2' / 'A2_S' / f'{constants.TAD_DATASET}_s8_nexar'
KNN      = V2_CACHE / 'knn' / f'{DATASET}_{C}' / constants.KNN_CACHE_FILENAME
T2       = constants.DATA_ROOT / T2NAME
T2_CENSUS = T2 / 'counts' / 'resolved_census.json'
ANNOT    = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME / constants.DADA_ORIGIN_ANNOTATION_FILENAME
TAD      = constants.DATA_ROOT / constants.TAD_DATASET
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = Path(DRIVE) / 'cache' / 'clip' / 'DoTA_s1_ncc'
ZS_ROOT  = constants.OUTPUT_ROOT / constants.ZS_OUTPUT_DIRNAME
A2_RUNS  = V2_OUT / constants.NEXAR_FULL_RUNS_DIR / 'A2'
# W0 = v2 Nexar A2 s2024, same code path and seed (ZS_TAD_DADA.md §2: DoTA-CAP-dev 0.8110072 in both).
# Both checkpoints are gone (KIM_RUN.md §4 dated note): W0 survives as a config (the K-rows' config diff) and
# as saved reads (a printed row). Nothing here loads W0's weights.
W0_CONFIGS = [constants.OUTPUT_ROOT / constants.WU_OUTPUT_DIRNAME / W0 / f's{SEED}' / 'config.yaml', A2_RUNS / f's{SEED}' / 'config.yaml']
W0_CONFIG = next((p for p in W0_CONFIGS if p.exists()), None)
assert W0_CONFIG is not None, f'no W0 config: upload the repo\'s outputs/v_final/wu/W0/s{SEED}/config.yaml to {W0_CONFIGS[0]}'
print(f'W0 config: {W0_CONFIG}')
W0_READ = {'dota': constants.OUTPUT_ROOT / constants.WU_OUTPUT_DIRNAME / 'eval' / 'dota' / CAP_DEV / W0,
           'dada': ZS_ROOT / 'eval' / 'dada' / W0,
           'tad':  ZS_ROOT / 'eval' / 'tad' / W0}            # holds s2024/
ROOT     = constants.OUTPUT_ROOT / constants.KIM_OUTPUT_DIRNAME    # Thesis-Final/outputs/kim
REPORTS  = ROOT / 'REPORTS'
RUN_DIR  = {a: ROOT / a / f's{SEED}' for a in constants.KIM_ARMS}
EVAL     = {s: ROOT / 'eval' / s for s in ('dota', 'dada', 'tad')}
LOCAL    = Path('/content/kim')
for p in (REPORTS, LOCAL):
    p.mkdir(parents=True, exist_ok=True)

PLAN = json.loads(BUILD.read_text())['plan'][C]
STEPS, EPOCHS = PLAN['steps'], PLAN['epochs']     # 1,749 = 33 x 53
for name, path in (('Nexar_window', DATA), ('Nexar A2 bake', A2_NEXAR), ('DADA A2 bake', A2_DADA),
                   ('DoTA-CAP A2 bake', A2_DOTA), ('TAD A2 bake', A2_TAD), ('KNN', KNN),
                   ('T2 meta', T2 / constants.META_FILENAME), ('T2 census', T2_CENSUS), ('DADA annotation', ANNOT),
                   ('TAD labels', TAD / constants.FRAME_LABELS_TEST_FILENAME), ('DoTA s1', DOTA_S1)):
    assert path.exists(), f'missing {name}: {path}'
    print(f'  {name:18s} {path}')
print(f'steps {STEPS} ({EPOCHS} epochs) | runs/reads -> {ROOT}')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn openpyxl xlrd
python -c "import av, einops, faiss, sklearn, openpyxl; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 5-9 need one'
df -h /content | tail -1

In [ ]:
import pickle
import shutil
import subprocess
import time

import numpy as np
import torch
import yaml

from core.config import Config                       # noqa: E402
from core.data.v2_splits import load_split           # noqa: E402
from core.tools.dada_whole_eval import test_sources  # noqa: E402
from core.tools.kim_pilot import first_mask_fits     # noqa: E402
from core.tools.v5_stage1 import _flat               # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    '''Stream a child's output here (and to ``log``); raise with its tail on failure.'''
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    '''Copy a Drive dir (only ``ids`` + json manifests if given) or file to VM disk once (bit-identical; C2).'''
    t0 = time.time()
    if src.is_file():
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
        return dst
    dst.mkdir(parents=True, exist_ok=True)
    if ids is None:
        for f in src.iterdir():
            if f.is_file() and not (dst / f.name).exists():
                shutil.copy2(f, dst / f.name)
    else:
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for pattern in ('*.json', '*.npz'):
            for m in src.glob(pattern):
                shutil.copy2(m, dst / m.name)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    '''VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10).'''
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def flush_drive():
    t0 = time.time()
    drive.flush_and_unmount()
    drive.mount('/content/drive')
    print(f'Drive flushed and remounted in {time.time() - t0:.0f}s')


def ckpt_step(path):
    '''The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step).'''
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None


def is_verified(drive_dir):
    rec, ckpt = drive_dir / 'verified.json', drive_dir / 'checkpoint_last.pt'
    if rec.exists() and ckpt.exists():
        r = json.loads(rec.read_text())
        return r['step'] == STEPS and r['ckpt_bytes'] == ckpt.stat().st_size
    return False


# ids and the preflight: nothing below runs on a wrong corpus
DADA_IDS = test_sources(T2)
CAP_IDS = sorted(load_split(CAP_DEV))
TAD_IDS = sorted(json.loads((TAD / constants.FRAME_LABELS_TEST_FILENAME).read_text()))
assert len(DADA_IDS) == 370, len(DADA_IDS)
assert len(TAD_IDS) == constants.V2_TADZ_VIDEOS, len(TAD_IDS)
for name, d, ids in (('DADA', A2_DADA, DADA_IDS), ('DoTA-CAP', A2_DOTA, CAP_IDS), ('TAD', A2_TAD, TAD_IDS)):
    miss = [v for v in ids if not (d / f'{v}.npy').exists()]
    assert not miss, f'{len(miss)} {name} videos have no A2 bake: {miss[:5]}'
for name, d in (('Nexar', A2_NEXAR), ('DADA', A2_DADA), ('DoTA-CAP', A2_DOTA), ('TAD', A2_TAD)):
    m = json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())
    assert (m['crn'], m['motion']) == (constants.V2_OFF, ENC), f'{name} bake is {m}'
same = {json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())['train_ids_sha1']
        for d in (A2_NEXAR, A2_DADA, A2_DOTA, A2_TAD)}
assert len(same) == 1, f'the bakes carry different statistics: {same}'
w0_cfg = _flat(yaml.safe_load(W0_CONFIG.read_text()))
assert w0_cfg['model.score_head_kernel'] == constants.DADA_ORIGIN_SCORE_HEAD_KERNEL, w0_cfg['model.score_head_kernel']
assert w0_cfg['dvs.theta'] < 1 and not w0_cfg.get('kim.enabled', False), 'W0 is not the DVS-on host'
L_DEFS = LOCAL / 'nexar_defs'
L_DEFS.mkdir(parents=True, exist_ok=True)
for f in DATA.glob('*.json'):               # defs.json etc.: the definitions the arms trained with
    shutil.copy2(f, L_DEFS / f.name)
print(f'preflight OK: {len(DADA_IDS)} DADA, {len(CAP_IDS)} DoTA-CAP-dev, {len(TAD_IDS)} TAD videos; shared statistics')

## 4. Tests on the KIM code (CPU) — **raises if anything fails: stop and paste the output**

In [ ]:
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_kim.py',
     'core/tests/test_kim_tools.py', 'core/tests/test_wu_arms.py'])

## 4b. W0's saved reads (no GPU)

The read-outs print W0 beside K0. Its checkpoint is gone, so a missing read **cannot** be re-read: it has to be
uploaded from the repo's `outputs/v_final/` to the same relative path under `Thesis-Final/outputs/`.

In [ ]:
L_T2 = LOCAL / 'T2_meta'
L_T2.mkdir(parents=True, exist_ok=True)
shutil.copy2(T2 / constants.META_FILENAME, L_T2 / constants.META_FILENAME)
L_TAD = LOCAL / 'TAD'
L_TAD.mkdir(parents=True, exist_ok=True)
for f in TAD.iterdir():                     # json / txt only -- never a frame folder
    if f.is_file() and f.suffix in ('.json', '.txt'):
        shutil.copy2(f, L_TAD / f.name)


def read_arm(arm):
    # one arm's reads with W0's protocols, into outputs/kim/eval/<set>/<arm>; skips what exists
    ckpt = RUN_DIR[arm] / 'checkpoint_last.pt'
    wrote = False
    if not (EVAL['dota'] / arm / 'clip_aucs.json').exists():
        local = LOCAL / 'dota' / arm
        run([sys.executable, '-m', 'core.tools.protocol_b_eval', '--run', f's{SEED}', ckpt,
             '--input-dir', stage(A2_DOTA, LOCAL / 'bake_dota', CAP_IDS), '--s1-dir', stage(DOTA_S1, LOCAL / 'dota_s1', CAP_IDS),
             '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
             '--data-dir', stage(DOTA / 'labels_s8', LOCAL / 'dota_labels'), '--split', CAP_DEV,
             '--out-dir', local, '--device', 'auto'], log=LOCAL / f'dota_{arm}.log')
        publish(local, EVAL['dota'] / arm)
        wrote = True
    if not (EVAL['dada'] / arm / 'clip_aucs.json').exists():
        local = LOCAL / 'dada' / arm
        run([sys.executable, '-m', 'core.tools.dada_whole_eval', '--run', f's{SEED}', ckpt, '--t2-dir', L_T2,
             '--annotation', ANNOT, '--census', T2_CENSUS, '--input-dir', stage(A2_DADA, LOCAL / 'bake_dada', DADA_IDS),
             '--data-dir', L_DEFS, '--out-dir', local], log=LOCAL / f'dada_{arm}.log')
        publish(local, EVAL['dada'] / arm)
        wrote = True
    tad_out = EVAL['tad'] / arm / f's{SEED}'
    if not (tad_out / 'results.json').exists():
        local = LOCAL / 'tad' / arm / f's{SEED}'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.evaluate', '--ckpt', ckpt, '--set', f'data.dataset={constants.TAD_DATASET}',
             '--data-dir', L_TAD, '--clip-dir', stage(A2_TAD, LOCAL / 'bake_tad', TAD_IDS),
             '--score-norm', 'auto', '--save-scores', '--output-dir', local], log=LOCAL / f'tad_{arm}.log')
        publish(local, tad_out)
        wrote = True
    if wrote:
        flush_drive()
    dota = json.loads((EVAL['dota'] / arm / 'protocol_b_readout.json').read_text())
    dada = json.loads((EVAL['dada'] / arm / 'dada_readout.json').read_text())
    tad = json.loads((tad_out / 'results.json').read_text())
    print(f"{arm}: DoTA-CAP-dev macro {dota['macro']['mean']:.4f} | DADA macro {dada['macro']['mean']:.4f} "
          f"| TAD macro {tad['auc_macro']:.4f}")


W0_NEEDS = {'dota': ('clip_aucs.json', 'protocol_b_readout.json'), 'dada': ('clip_aucs.json', 'dada_readout.json')}
missing = [W0_READ[s] / f for s, files in W0_NEEDS.items() for f in files if not (W0_READ[s] / f).exists()]
tad_scores = W0_READ['tad'] / f's{SEED}' / 'scores'
n_tad = len(list(tad_scores.glob('*.npz'))) if tad_scores.is_dir() else 0
if n_tad != constants.V2_TADZ_VIDEOS:          # C10: a folder is not evidence of its files
    missing.append(tad_scores)
assert not missing, ("W0's reads cannot be redone (no checkpoint). Upload from the repo's outputs/v_final/ to the same "
                     f"path under {constants.OUTPUT_ROOT}: {[str(p.relative_to(constants.OUTPUT_ROOT)) for p in missing]}")
print(f'W0 reads OK: {W0_READ}')


## 5. Train K0 and K0k9 (GPU)

W0's recipe (`wu_arms.ipynb`) with DVS off; K0k9 adds `model.score_head_kernel=9`. **Watch the log: past 2 s/step,
stop and paste it.** Then J10 and the config diff against W0 (only `dvs.theta*`, the arm's own keys, and keys that
did not exist when W0 trained, at their defaults).

In [ ]:
DEFAULTS = _flat(Config().to_dict())


EDGE_SETS = []                        # ['kim.edge_mask=true'] once step 9 reads a KP3 failure (KIM_RUN.md §5.2)


def arm_sets(arm, kernel):
    extra = {K0: [], K0K9: [], K2: ['kim.enabled=true', *EDGE_SETS], K5: ['kim.enabled=true', 'kim.prototypes=true', *EDGE_SETS]}[arm]
    return [f'model.score_head_kernel={kernel}', 'dvs.theta=1.0', 'dvs.theta_ego=1.0', *extra]


def train_cmd(arm, kernel, out_dir, data, knn):
    cmd = [sys.executable, '-m', 'core.train',
           '--set', 'train.stage=2', '--set', f'train.seed={SEED}',
           '--set', f'train.num_epochs={EPOCHS}', '--set', 'train.amp=true',
           '--set', f'data.dataset={DATASET}',
           '--set', f'loss.mil_topk_pct={constants.DADA_ORIGIN_MIL_TOPK_PCT}',
           '--set', 'kip.enabled=false', '--set', 'loss.dvs_anchor_mode=span',
           '--set', 'data.is_egocentric=false',
           '--set', f'v2.motion={ENC}', '--set', f'model.motion_dim={constants.V5_GRID_DIM}']
    for s in arm_sets(arm, kernel):
        cmd += ['--set', s]
    return cmd + ['--data-dir', data, '--clip-dir', A2_LOCAL, '--output-dir', out_dir, '--knn-cache', knn]


def train_arm(arm, kernel):
    drive_dir, local = RUN_DIR[arm], LOCAL / 'runs' / arm
    if is_verified(drive_dir):
        print(f'{arm} s{SEED}: verified on Drive -- not retrained')
        return
    assert torch.cuda.is_available(), 'torch sees no GPU -- train.device=auto would silently train on CPU'
    local.mkdir(parents=True, exist_ok=True)
    resume = ['--resume', local / 'checkpoint_last.pt'] if (local / 'checkpoint_last.pt').exists() else []
    t0 = time.time()
    run(train_cmd(arm, kernel, local, DATA_LOCAL, KNN_LOCAL) + resume, log=LOCAL / f'train_{arm}.log')
    last = metrics_last(local / 'metrics.jsonl')
    assert ckpt_step(local / 'checkpoint_last.pt') == last['global_step'] == STEPS, f'{arm}: J10 locally'
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        publish(local / name, drive_dir / name)
    flush_drive()
    assert ckpt_step(drive_dir / 'checkpoint_last.pt') == STEPS, f'{arm}: Drive copy is not step {STEPS}'
    (LOCAL / 'verified.json').write_text(json.dumps(
        {'step': STEPS, 'ckpt_bytes': (drive_dir / 'checkpoint_last.pt').stat().st_size}))
    publish(LOCAL / 'verified.json', drive_dir / 'verified.json')
    print(f'{arm}: {(time.time() - t0) / 60:.0f} min | mil {last.get("mil")} | kim_mil {last.get("kim_mil")} '
          f'| W_out {last.get("kim_w_out_norm")} | proto cos {last.get("kim_proto_cos")}')


def check_run(arm, kernel):
    d = RUN_DIR[arm]
    payload = torch.load(d / 'checkpoint_last.pt', map_location='cpu', weights_only=False)
    assert payload['global_step'] == metrics_last(d / 'metrics.jsonl')['global_step'] == STEPS, f'{arm}: J10'
    got = _flat(yaml.safe_load((d / 'config.yaml').read_text()))
    allowed = {k.split('=')[0] for k in arm_sets(arm, kernel)}
    diff = {}
    for k in set(w0_cfg) | set(got):
        if w0_cfg.get(k) == got.get(k) or k in allowed:
            continue
        if k not in w0_cfg and got.get(k) == DEFAULTS.get(k):
            continue                      # a key added after W0 trained, left at its default
        diff[k] = (w0_cfg.get(k), got.get(k))
    assert not diff, f'{arm} differs from W0 beyond its own keys: {diff}'
    has_kim = any(k.startswith('kim.') for k in payload['model'])
    assert has_kim == (arm in (K2, K5)), f'{arm}: KIM weights present = {has_kim}'
    assert len(payload['optimizer']['param_groups']) == 1, f'{arm}: one AdamW group (D13)'
    first = {a: json.loads((RUN_DIR[a] / 'metrics.jsonl').read_text().splitlines()[0]) for a in (arm, K0)}
    print(f'{arm}: J10 {STEPS}, config == W0 + {sorted(allowed)} | step-1 mil {first[arm]["mil"]:.5f} vs K0 {first[K0]["mil"]:.5f}')


A2_LOCAL = stage(A2_NEXAR, LOCAL / 'stage' / 'a2_nexar')
DATA_LOCAL = stage(DATA, LOCAL / 'stage' / 'data')
KNN_LOCAL = stage(KNN, LOCAL / 'stage' / 'knn.npz')
K3_ = constants.DADA_ORIGIN_SCORE_HEAD_KERNEL
for arm, kernel in ((K0, K3_), (K0K9, constants.KIM_HOST_KERNEL_K9)):
    train_arm(arm, kernel)
    check_run(arm, kernel)

## 6. Zero-shot reads (GPU): DoTA-CAP-dev protocol B, DADA-2000 whole video, TAD (W0's protocols)

In [ ]:
assert torch.cuda.is_available(), 'torch sees no GPU'
for arm in (K0, K0K9):
    read_arm(arm)

## 7. Read-out 1 — K0k9 vs K0 → **kernel decision (§3.1)**; paste `REPORTS/kim_readout_host.md`

In [ ]:
def readout_cmd(rows, control, out_name, pilot=None):
    cmd = [sys.executable, '-m', 'core.tools.kim_readout', '--control', control,
           '--dota', W0, W0_READ['dota'], '--dada', W0, W0_READ['dada'], '--tad', W0, W0_READ['tad'],
           '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt']
    for arm in rows:
        cmd += ['--dota', arm, EVAL['dota'] / arm, '--dada', arm, EVAL['dada'] / arm,
                '--tad', arm, EVAL['tad'] / arm, '--run', arm, RUN_DIR[arm]]
    if pilot is not None:
        cmd += ['--pilot', pilot]
    local = LOCAL / out_name
    run(cmd + ['--out-dir', local])
    publish(local / 'kim_readout.md', REPORTS / f'{out_name}.md')
    publish(local / 'kim_readout.json', REPORTS / f'{out_name}.json')
    return json.loads((REPORTS / f'{out_name}.json').read_text())


host = readout_cmd([K0, K0K9], K0, 'kim_readout_host')
print((REPORTS / 'kim_readout_host.md').read_text())
KERNEL_9 = host['decisions'].get('kernel') == 'host = kernel 9 for every K row'
HOST_KERNEL = constants.KIM_HOST_KERNEL_K9 if KERNEL_9 else constants.DADA_ORIGIN_SCORE_HEAD_KERNEL
CONTROL = K0K9 if KERNEL_9 else K0
print(f'\n>>> §3.1: host kernel {HOST_KERNEL}; control for K2 / K5 = {CONTROL}')

## 8. Pilot KP1–KP4 (GPU, inference only, on the decided control) → **paste `REPORTS/pilot/kim_pilot.md` to Claude**

Labels and the host's scores come from step 6's reads of that control; inputs from the bakes those reads used.
The decisions (KP1 input, KP3 edge mask, KP4 / H6) are fixed in `KIM_RUN.md` §5, §5.1. **If KP1 picks an input other
than `vt`, stop here and tell Claude before step 9** (the K2 / K5 build changes; K0 / K0k9 do not). 8b re-reads the same
pilot with the KP3 split (§5.2); a KP3 failure then sets `kim.edge_mask` (first valid pair, §5.3) on K5.

In [ ]:
PILOT = REPORTS / 'pilot'
PILOT_EDGES = REPORTS / 'pilot_edges'


def pilot_run(out_dir):
    L_HOST = {s: stage(EVAL[s] / CONTROL, LOCAL / 'host_read' / s) for s in ('dota', 'dada')}
    stage(EVAL['tad'] / CONTROL / f's{SEED}' / 'scores', LOCAL / 'host_read' / 'tad' / 'scores')
    L_BAKE = {'dota': stage(A2_DOTA, LOCAL / 'bake_dota', CAP_IDS), 'dada': stage(A2_DADA, LOCAL / 'bake_dada', DADA_IDS),
              'tad': stage(A2_TAD, LOCAL / 'bake_tad', TAD_IDS)}
    run([sys.executable, '-m', 'core.tools.kim_pilot', '--name', CONTROL, '--ckpt', RUN_DIR[CONTROL] / 'checkpoint_last.pt',
         '--set', CAP_DEV, 'native', L_HOST['dota'], L_BAKE['dota'],
         '--set', 'dada', 'rows', L_HOST['dada'], L_BAKE['dada'],
         '--set', 'tad', 'rows', LOCAL / 'host_read' / 'tad', L_BAKE['tad'],
         '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
         '--out-dir', out_dir], log=LOCAL / f'{out_dir.name}.log')


if (PILOT / 'kim_pilot.json').exists():
    host = json.loads((PILOT / 'kim_pilot.json').read_text()).get('host')
    assert host == CONTROL, f'a pilot of {host} is on Drive but the control is {CONTROL} (KIM_RUN.md §5.1): ask Claude'
else:
    assert torch.cuda.is_available(), 'torch sees no GPU'
    pilot_run(LOCAL / 'pilot')
    publish(LOCAL / 'pilot', PILOT)
    flush_drive()
print((PILOT / 'kim_pilot.md').read_text())

# 8b: the same pilot with the KP3 split (KIM_RUN.md §5.2)
if not (PILOT_EDGES / 'kim_pilot.json').exists():
    assert torch.cuda.is_available(), 'torch sees no GPU'
    pilot_run(LOCAL / 'pilot_edges')
    publish(LOCAL / 'pilot_edges', PILOT_EDGES)
    flush_drive()
print((PILOT_EDGES / 'kim_pilot.md').read_text())
first = json.loads((PILOT / 'kim_pilot.json').read_text())
again = json.loads((PILOT_EDGES / 'kim_pilot.json').read_text())
assert again['host'] == CONTROL, f"pilot_edges read {again['host']}, the control is {CONTROL}"
for key in ('KP1', 'KP2', 'KP3', 'KP4'):
    assert first['decisions'][key]['verdict'] == again['decisions'][key]['verdict'], f'{key}: the re-read changed the verdict'
assert first['decisions']['input'] == again['decisions']['input'], 'the re-read changed the KP1 input'
dec = again['decisions']
FIRST_FITS = first_mask_fits(again['sets'])         # from the numbers: an older json has no such field
print(f">>> input {dec['input']} | KP3 {dec['KP3']['verdict']} | first-pair mask fits: {FIRST_FITS}")
if dec['input'] != 'vt':
    print('>>> KP1 changed the K2 / K5 input: do NOT run step 9, tell Claude.')
if dec['KP3']['verdict'] == 'fail' and not FIRST_FITS:
    print('>>> the first pair no longer carries the excess (§5.3): do NOT run step 9, tell Claude.')

## 9. Train **K5** on the decided host, then read it (GPU)

Refused if step 8b flagged KP1, or if the KP3 excess is no longer on the first pair. A KP3 failure adds `kim.edge_mask=true` (first valid pair, §5.3). K2 waits for K5 (§6.1). K5's NKC and prototype updates start at step 219 (D11).
If the host moved to kernel 9, K2 / K5 train with kernel 9 and K0k9 is their control.

In [ ]:
dec = json.loads((PILOT_EDGES / 'kim_pilot.json').read_text())['decisions']
assert dec['input'] == 'vt', 'KP1 changed the K2 / K5 input (KIM_RUN.md §5): ask Claude before training them'
if dec['KP3']['verdict'] == 'fail':
    assert FIRST_FITS, 'the KP3 excess is no longer on the first pair (KIM_RUN.md §5.3): ask Claude before training'
    EDGE_SETS[:] = ['kim.edge_mask=true']
print(f'K2 / K5 host kernel {HOST_KERNEL}, extra sets {EDGE_SETS}')
for arm in (K5,):                  # KIM_RUN.md §6.1: K2 only if K5 passes
    train_arm(arm, HOST_KERNEL)
    check_run(arm, HOST_KERNEL)
    read_arm(arm)

## 10. Read-out 2 → **stop rule (§6)**; paste `REPORTS/kim_readout.md` to Claude

In [ ]:
full = readout_cmd([K0, K0K9, K5], CONTROL, 'kim_readout', pilot=PILOT_EDGES / 'kim_pilot.json')
print((REPORTS / 'kim_readout.md').read_text())
print(f">>> stop rule: {full['decisions']['stop_rule']}")
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
drive.mount('/content/drive')
print('Drive flushed -- paste REPORTS/pilot_edges/kim_pilot.md and REPORTS/kim_readout.md to Claude')